<div style="background:linear-gradient(125deg,#071827,#154B51);color:#F8F3E5;padding:40px;border-radius:20px;">
<p style="color:#F4C95D;letter-spacing:.17em;font-size:12px;">PLAYGROUND S6E9 · UNSUPERVISED DIVERSITY</p>
<h1 style="color:#F8F3E5;font-size:40px;line-height:1.08;">Fifteen Teachers, Fewer Voices</h1>
<p style="font-size:20px;">Rank correlation · cluster medoids · five frozen-fold readouts</p>
<p style="color:#3DD5C7;">Count independent orderings, not filenames</p>
</div>

### The idea

Public [OOF libraries](https://www.kaggle.com/code/najiama/s6e9-electric-vehicle-oof-cv-0-94618-lb-0-94633) and [rank ensembles](https://www.kaggle.com/code/amanatar/s6e9-smart-weighted-rank-ensemble) make it easy to average many files. Our notebook 23 found why provenance matters: a pair of OOF/test files is not proof of independent model lineage. Here we ask a narrower question that needs **no target-trained meta model**: if two teachers produce nearly the same ordering, should they receive two votes?

## 01 · Inputs

Attach the competition, `najiama/s6e9-oof`, and `megayak/s6e9-six-feature-views-oof-library`. CPU and Internet Off. This notebook reads the same 15 OOF/test pairs as notebook 23. It does **not** require its output or the Zoom archive. Missing or misaligned files fail before a submission is written.

In [ ]:
from pathlib import Path
import hashlib, io, json, zipfile, time
STARTED = time.perf_counter()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from IPython.display import HTML, display
ID,TARGET="id","Will_Buy_EV"; BUILD_ID="S6E9-FIFTEEN-TEACHERS-FEWER-VOICES-20260924"
ROOT=Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd(); OUTPUT=Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs/cluster-medoids")
OUTPUT.mkdir(parents=True,exist_ok=True); print("BUILD ID:",BUILD_ID)
def one(pattern,contains=None):
    hits=sorted({p for p in ROOT.rglob(pattern) if p.is_file()})
    if contains:
        narrowed=[p for p in hits if contains.lower() in str(p).lower()]
        if narrowed: hits=narrowed
    if len(hits)!=1: raise RuntimeError(f"Expected one {pattern}; found {len(hits)}: {hits[:8]}")
    return hits[0]
def rank01(x): return rankdata(np.asarray(x),method="average")/len(x)
def sha_rank(x): return hashlib.sha256(np.asarray(rankdata(x,method="average"),dtype=np.float64).tobytes()).hexdigest()

In [ ]:
train=pd.read_csv(one("train.csv",contains="playground-series-s6e9")); sample=pd.read_csv(one("sample_submission.csv",contains="playground-series-s6e9")); y=train[TARGET].map({"No":0,"Yes":1}).astype(int)
assert train[ID].is_unique and sample[ID].is_unique
CORE={
"01 Blend":("01_blend_oof.csv","01_submission.csv"),"Pure LGBM v1":("Pure LGBM_V1_oof.csv","Pure LGBM_V1_test.csv"),"Pure LGBM v3":("Pure LGBM_V3_oof.csv","Pure LGBM_V3_test.csv"),"Sergey LGBM":("Sergey_LGBM_oof.csv","Sergey_LGBM_submission.csv"),"XGB triple TE 10f":("XGBoost_Triple_TE_10folds_oof.csv","XGBoost_Triple_TE_10folds_test.csv"),"XGB triple TE 5f":("XGBoost_Triple_TE_5folds_oof.csv","XGBoost_Triple_TE_5folds_test.csv")}
SIX=["A_lgbm_triple_te_digits_3seed","B_xgb_on_A_features","C_no_digits_windows_lift_sm2_30_300","D_no_exact_key_ladder_windows","E_ladder25_250_2500_lift_sm5_50_500","F_exact_rate_as_init_score","ensemble"]
REAL=["G_realmlp_10fold","G_realmlp_3seed"]
def prediction(path,ids):
    frame=pd.read_csv(path); cols=[c for c in frame if c!=ID]
    assert ID in frame and frame[ID].is_unique and len(cols)==1
    values=pd.DataFrame({ID:ids}).merge(frame[[ID,cols[0]]],on=ID,how="left",validate="one_to_one")[cols[0]].to_numpy(float)
    assert len(values)==len(ids) and np.isfinite(values).all(); return values

def aligned(frame,ids,column):
    assert ID in frame and frame[ID].is_unique and column in frame
    values=pd.DataFrame({ID:ids}).merge(frame[[ID,column]],on=ID,how="left",validate="one_to_one")[column].to_numpy(float)
    assert len(values)==len(ids) and np.isfinite(values).all(); return values

oof={}; test_pred={}; lineage={}
for name,(of,tf) in CORE.items():
    oof[name]=prediction(one(of),train[ID]); test_pred[name]=prediction(one(tf),sample[ID]); lineage[name]="separate OOF/test pair; fold vector absent"
six_oof,six_test=pd.read_csv(one("oof_six_views.csv")),pd.read_csv(one("test_six_views.csv")); real_oof,real_test=pd.read_csv(one("oof_realmlp_g.csv")),pd.read_csv(one("test_realmlp_g.csv"))
assert np.array_equal(six_oof[TARGET].astype(int),y) and np.array_equal(real_oof[TARGET].astype(int),y)
for name in SIX:
    oof[name]=aligned(six_oof,train[ID],name); test_pred[name]=aligned(six_test,sample[ID],name); lineage[name]="shared table; target checked; fold vector absent"
for name in REAL:
    oof[name]=aligned(real_oof,train[ID],name); test_pred[name]=aligned(real_test,sample[ID],name); lineage[name]="shared table; target checked; fold vector absent"
assert len(oof)==15

## 02 · Two votes, one experiment

The control gives all 15 teachers equal weight after ranking. The candidate clusters teachers with Spearman correlation at least about 0.998, chooses each cluster's medoid using **prediction correlation only**, then gives each cluster one equal vote. The threshold is fixed before seeing target scores. No class label chooses the medoid or the cluster weight. This changes one material thing: the influence of redundant teachers.

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.model_selection import StratifiedKFold
import platform
import time

SEED = 42
BUILD_ID = "S6E9-FIFTEEN-TEACHERS-FEWER-VOICES-20260924"
names = sorted(oof)
oof_rank = pd.DataFrame({name: rank01(oof[name]) for name in names})
test_rank = pd.DataFrame({name: rank01(test_pred[name]) for name in names})
assert len(names) == 15 and oof_rank.shape == (len(train), 15)
assert test_rank.shape == (len(sample), 15)
corr = oof_rank.corr(method="spearman")
distance = np.clip(1 - corr.to_numpy(), 0, 2)
np.fill_diagonal(distance, 0)
tree = linkage(squareform(distance, checks=True), method="average")
labels = fcluster(tree, t=.002, criterion="distance")
clusters = {}
for name, label in zip(names, labels):
    clusters.setdefault(int(label), []).append(name)
medoids = []
for label, members in sorted(clusters.items()):
    chosen = sorted(members, key=lambda n: (-float(corr.loc[n, members].mean()), n))[0]
    medoids.append(chosen)
cluster_table = pd.DataFrame([
    {"cluster": label, "members": ", ".join(members), "size": len(members),
     "medoid": sorted(members, key=lambda n: (-float(corr.loc[n, members].mean()), n))[0]}
    for label, members in sorted(clusters.items())
])
assert len(medoids) == len(set(medoids)) and set(medoids).issubset(names)
display(cluster_table)
print(f"{len(names)} files → {len(medoids)} correlation clusters")

## 03 · Same rows, same folds

All scores are recomputed on the competition target. The five displayed fold slices use the repository's frozen v1 `StratifiedKFold(5, shuffle=True, seed=42)` assignment. The base predictions were supplied by others without exact training-fold lineage, so these scores are **OOF diagnostics**, not a proof that a new supervised stack would generalize. The candidate never fits on labels, which avoids the most direct meta-fold leakage.

In [ ]:
control_oof = oof_rank.mean(axis=1).to_numpy(float)
candidate_oof = oof_rank[medoids].mean(axis=1).to_numpy(float)
control_test = test_rank.mean(axis=1).to_numpy(float)
candidate_test = test_rank[medoids].mean(axis=1).to_numpy(float)
assert all(np.isfinite(v).all() for v in (control_oof, candidate_oof, control_test, candidate_test))
fold = np.empty(len(train), dtype=np.int8)
for k, (_, valid) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(train, y)):
    fold[valid] = k
assert set(fold) == set(range(5))
rows = []
for k in range(5):
    mask = fold == k
    a = roc_auc_score(y[mask], control_oof[mask])
    b = roc_auc_score(y[mask], candidate_oof[mask])
    rows.append({"fold": k, "equal_15_auc": a, "medoid_auc": b, "delta": b-a})
fold_table = pd.DataFrame(rows)
control_auc = roc_auc_score(y, control_oof)
candidate_auc = roc_auc_score(y, candidate_oof)
delta = candidate_auc - control_auc
accepted = bool(delta >= .000020 and (fold_table.delta > 0).sum() >= 4)
print(f"Equal-15 OOF {control_auc:.7f} · cluster-medoid OOF {candidate_auc:.7f} · Δ {delta:+.7f}")
print(f"Predeclared gate +0.000020 and 4/5: {accepted}")
display(fold_table.style.format({"equal_15_auc": "{:.7f}", "medoid_auc": "{:.7f}", "delta": "{:+.7f}"}))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(corr.where(~np.eye(len(corr), dtype=bool)).stack(), bins=25, color="#267E87")
axes[0].axvline(.998, color="#D69228", linestyle="--")
axes[0].set(title="Pairwise teacher Spearman", xlabel="Correlation", ylabel="Pairs")
axes[1].bar(fold_table.fold.astype(str), fold_table.delta, color="#D69228")
axes[1].axhline(0, color="#405260", linewidth=1)
axes[1].set(title="Medoid-minus-equal AUC by frozen fold", xlabel="Fold", ylabel="AUC delta")
plt.tight_layout(); plt.show()

## 04 · Candidate and reproducibility files

The selected `submission.csv` is the medoid average only if its prespecified OOF gate passes; otherwise it is the equal-15 control. We also save both named candidates, row-aligned OOF predictions, the cluster ledger, and a JSON report. A submitted candidate needs its own public score; the notebook never labels it 0.94656 by inheritance.

In [ ]:
import json
from hashlib import sha256

def submission(values, path):
    frame = pd.DataFrame({ID: sample[ID], TARGET: np.asarray(values, dtype=float)})
    assert frame.columns.tolist() == [ID, TARGET] and frame[ID].is_unique
    assert frame[TARGET].between(0, 1).all() and np.isfinite(frame[TARGET]).all()
    frame.to_csv(path, index=False)
    return sha256(path.read_bytes()).hexdigest()

control_sha = submission(control_test, OUTPUT / "submission_equal_15.csv")
candidate_sha = submission(candidate_test, OUTPUT / "submission_cluster_medoids.csv")
selected = candidate_test if accepted else control_test
selected_sha = submission(selected, OUTPUT / "submission.csv")
pd.DataFrame({ID: train[ID], "fold": fold, TARGET: y,
              "equal_15": control_oof, "cluster_medoids": candidate_oof}).to_csv(
    OUTPUT / "oof_cluster_medoids.csv", index=False
)
cluster_table.to_csv(OUTPUT / "cluster_medoid_ledger.csv", index=False)
report = {
    "build_id": BUILD_ID, "metric": "roc_auc", "fold_version": "v1", "seed": SEED,
    "code_revision": BUILD_ID, "duration_seconds": time.perf_counter() - STARTED,
    "environment": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__},
    "teachers": names, "cluster_threshold_spearman": .998, "cluster_count": len(medoids),
    "medoids": medoids, "equal_15_oof_auc": control_auc,
    "medoid_oof_auc": candidate_auc, "delta": delta,
    "folds": fold_table.to_dict("records"), "gate_passed": accepted,
    "selected": "cluster_medoids" if accepted else "equal_15",
    "submission_sha256": selected_sha, "control_sha256": control_sha,
    "candidate_sha256": candidate_sha, "public_score": None,
    "limitation": "Base OOF fold lineage is not published; no target-trained stack is fitted."
}
(OUTPUT / "cluster_medoid_report.json").write_text(json.dumps(report, indent=2))
display(HTML(f"""<div style="background:#071827;color:#F8F3E5;padding:25px;border-radius:16px;"><p style="color:#F4C95D;letter-spacing:.13em;">DIVERSITY EXPERIMENT</p><h2 style="color:#F8F3E5;">{len(names)} teachers → {len(medoids)} voices</h2>OOF Δ {delta:+.7f} · selected: {report["selected"]} · public score unknown.</div>"""))
print(sorted(p.name for p in OUTPUT.iterdir()))

### What we learned

If the medoid ensemble wins, redundant teachers were overweighted in a plain mean. If it loses, the apparently redundant rankings still carried useful information or the cluster threshold was too aggressive. The outcome belongs in the report either way; repeated threshold tuning against the same OOF would erase the point of the experiment.